# 06 - GiroSUS: ocupação de leitos do SUS em Goiás

Este notebook sustenta o produto **GiroSUS**, um painel que troca a pergunta *"quantas internações?"* por **"quem ocupa os leitos, por quanto tempo e quanto isso custa ao SUS?"**, olhando todas as causas de internação.

Ele calcula todos os números usados no material comercial (`dashboard/GiroSUS_painel_ocupacao_leitos.pdf`) e responde, em ordem, as 34 perguntas do painel, organizadas em três níveis:

- **Nível 1, o retrato:** quanto leito o SUS usou e quanto pagou.
- **Nível 2, onde e quem:** quais doenças, hospitais e municípios ocupam os leitos, e quanto recebem.
- **Nível 3, onde agir:** onde os pacientes ficam além do típico, quando a rede enche e quanto se ganharia.

**Base:** somente SIH/SUS (`data/silver/sih_multianual.parquet`). A base de população do IBGE entra apenas para dar nome aos municípios.

**Safra:** competências de janeiro a dezembro de 2025 (último ano completo). O histórico 2021 a jun/2026 é usado só no calendário de ocupação.

## 1. Configuração inicial

In [1]:
import sys
print(sys.executable)
%pip install pyarrow==25.0.1

c:\Users\USER\AppData\Local\Programs\Python\Python312\python.exe
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PASTA_SILVER = Path("../data/silver")
PASTA_RAW_SIH = Path("../data/raw/sih")

SAFRA = "2025"
MUNICIPIO_GOIANIA = "520870"
LIMITE_INTERNACAO_LONGA = 15  # dias

## 2. Carregamento das bases

In [3]:
df_sih = pd.read_parquet(PASTA_SILVER / "sih_multianual.parquet")
df_populacao = pd.read_parquet(PASTA_SILVER / "populacao_multianual.parquet")

print("SIH:", df_sih.shape)
print("População (usada só para nomes):", df_populacao.shape)

SIH: (2286755, 19)
População (usada só para nomes): (33422, 10)


## 3. Regras de negócio e safra

As regras seguem as dos notebooks 05 e a ficha técnica do produto:

- **Safra:** AIHs apresentadas nas competências de 2025 (`ANO_CMPT = 2025`). Competência é o mês em que o hospital apresentou a internação para pagamento.
- **Só AIH regular (`IDENT = 1`).** As AIHs de continuação (`IDENT = 5`) são a mesma internação cobrada em partes e ficariam contadas duas vezes.
- **Todas as causas de internação**, em hospitais localizados em Goiás.

In [4]:
df_safra_bruta = df_sih[df_sih["ANO_CMPT"].astype(str) == SAFRA].copy()
df = df_safra_bruta[df_safra_bruta["IDENT"].astype(str) == "1"].copy()

funil = pd.DataFrame({
    "etapa": [
        f"Registros das competências de {SAFRA}",
        "AIHs de continuação (IDENT = 5) retiradas",
        "Internações regulares no painel (IDENT = 1)",
    ],
    "quantidade": [
        len(df_safra_bruta),
        len(df_safra_bruta) - len(df),
        len(df),
    ],
})
funil

,etapa,quantidade
0,Registros das competências de 2025,467902
1,AIHs de continuação (IDENT = 5) retiradas,10499
2,Internações regulares no painel (IDENT = 1),457403


## 4. Preparação das colunas do produto

Todas as regras ficam aqui, em Python, para o Power BI só somar e dividir:

- `leito_dias`: dias de permanência (`DIAS_PERM`). Um paciente ocupando um leito por um dia.
- `valor_pago`: valor pago pelo SUS (`VAL_TOT`). **Não é o custo real do hospital.**
- `grupo_doenca`: capítulo da CID-10 pela primeira letra do diagnóstico principal.
- `tempo_tipico`: mediana de dias do mesmo procedimento (`PROC_REA`) na safra.
- `dias_acima_tipico`: dias além do tempo típico. É um sinal para investigar, não um desperdício por definição.
- `fl_fora_municipio`: paciente internado fora do município onde mora.

In [5]:
df["leito_dias"] = pd.to_numeric(df["DIAS_PERM"], errors="coerce").fillna(0)
df["valor_pago"] = pd.to_numeric(df["VAL_TOT"], errors="coerce").fillna(0)
df["dias_uti"] = pd.to_numeric(df["UTI_MES_TO"], errors="coerce").fillna(0)
df["fl_uti"] = df["dias_uti"] > 0

df["DIAG_PRINC"] = df["DIAG_PRINC"].astype("string").str.upper()
df["MUNIC_RES"] = df["MUNIC_RES"].astype("string").str.zfill(6)
df["MUNIC_MOV"] = df["MUNIC_MOV"].astype("string").str.zfill(6)
df["fl_fora_municipio"] = df["MUNIC_RES"] != df["MUNIC_MOV"]

df["DT_INTER"] = pd.to_datetime(df["DT_INTER"])
df["mes_competencia"] = df["MES_CMPT"].astype(int)

In [6]:
GRUPOS_CID = {
    "A": "Infecciosas", "B": "Infecciosas", "C": "Câncer",
    "E": "Endócrinas", "F": "Saúde mental", "G": "Sistema nervoso",
    "H": "Olho e ouvido", "I": "Circulatório", "J": "Respiratório",
    "K": "Digestivo", "L": "Pele", "M": "Osteomuscular",
    "N": "Geniturinário", "O": "Gravidez e parto", "P": "Perinatal",
    "Q": "Malformações", "R": "Sintomas mal definidos",
    "S": "Lesões e traumas", "T": "Lesões e traumas",
    "Z": "Contatos com serviço de saúde",
}


def classificar_grupo(cid):
    if pd.isna(cid) or len(cid) == 0:
        return "Outros"
    letra = cid[0]
    if letra == "D":
        numero = cid[1:3]
        return "Câncer" if numero.isdigit() and int(numero) < 50 else "Sangue"
    return GRUPOS_CID.get(letra, "Outros")


df["grupo_doenca"] = df["DIAG_PRINC"].map(classificar_grupo)

df["tempo_tipico"] = df.groupby("PROC_REA")["leito_dias"].transform("median")
df["dias_acima_tipico"] = (df["leito_dias"] - df["tempo_tipico"]).clip(lower=0)

df["faixa_duracao"] = pd.cut(
    df["leito_dias"],
    bins=[-1, 0, 3, 7, 15, 30, np.inf],
    labels=["0 dia", "1–3", "4–7", "8–15", "16–30", "> 30"],
)

In [7]:
# Nomes de municípios: única informação usada da base do IBGE
populacao_2025 = df_populacao[df_populacao["ano_referencia"] == 2025].copy()
populacao_2025["codigo_6"] = populacao_2025["codigo_ibge_7"].astype(str).str[:6]
NOME_MUNICIPIO = dict(zip(populacao_2025["codigo_6"], populacao_2025["municipio"]))

df["nome_municipio_residencia"] = df["MUNIC_RES"].map(NOME_MUNICIPIO).fillna(df["MUNIC_RES"])
df["nome_municipio_atendimento"] = df["MUNIC_MOV"].map(NOME_MUNICIPIO).fillna(df["MUNIC_MOV"])

In [8]:
TOTAL_INTERNACOES = len(df)
TOTAL_LEITOS_DIA = df["leito_dias"].sum()
TOTAL_VALOR = df["valor_pago"].sum()
PERMANENCIA_MEDIA = TOTAL_LEITOS_DIA / TOTAL_INTERNACOES


def resumo(grupo):
    tabela = df.groupby(grupo, observed=True).agg(
        internacoes=("N_AIH", "size"),
        leitos_dia=("leito_dias", "sum"),
        valor_pago=("valor_pago", "sum"),
        dias_acima_tipico=("dias_acima_tipico", "sum"),
    )
    tabela["pct_internacoes"] = tabela["internacoes"] / TOTAL_INTERNACOES * 100
    tabela["pct_leitos_dia"] = tabela["leitos_dia"] / TOTAL_LEITOS_DIA * 100
    tabela["pct_valor"] = tabela["valor_pago"] / TOTAL_VALOR * 100
    tabela["permanencia_media"] = tabela["leitos_dia"] / tabela["internacoes"]
    tabela["rs_por_leito_dia"] = tabela["valor_pago"] / tabela["leitos_dia"]
    tabela["rs_por_internacao"] = tabela["valor_pago"] / tabela["internacoes"]
    tabela["pct_acima_tipico"] = tabela["dias_acima_tipico"] / tabela["leitos_dia"] * 100
    return tabela.sort_values("leitos_dia", ascending=False)

## 5. Nível 1 · O retrato (página 1 do painel)

**Pergunta principal:** qual é o tamanho da ocupação de leitos do SUS em Goiás?

Perguntas 1 a 6: quantos leitos-dia, internações, valor pago, permanência média, R$ por leito-dia e pacientes internados num dia comum.

In [9]:
retrato = pd.Series({
    "1. Leitos-dia": TOTAL_LEITOS_DIA,
    "2. Internações": TOTAL_INTERNACOES,
    "2. Hospitais com internação": df["CNES"].nunique(),
    "3. Valor pago pelo SUS (R$)": TOTAL_VALOR,
    "4. Permanência média (dias)": PERMANENCIA_MEDIA,
    "5. R$ por leito-dia": TOTAL_VALOR / TOTAL_LEITOS_DIA,
})
retrato.to_frame("valor")

,valor
1. Leitos-dia,"1,835,227.00"
2. Internações,"457,403.00"
2. Hospitais com internação,260.00
3. Valor pago pelo SUS (R$),"726,059,090.02"
4. Permanência média (dias),4.01
5. R$ por leito-dia,395.62


### 5.1. Pacientes internados por dia (censo diário)

Para cada dia do ano conta-se quem já tinha entrado (`DT_INTER`) e ainda não tinha saído (`DT_SAIDA`). Usa-se o histórico inteiro da base silver para incluir quem entrou antes de 2025 e ainda estava internado.

O fim de dezembro sai subestimado: parte das internações ainda não havia sido apresentada para pagamento.

In [10]:
df_censo = df_sih[df_sih["IDENT"].astype(str) == "1"][["DT_INTER", "DT_SAIDA", "DIAG_PRINC"]].copy()
df_censo["entrada"] = pd.to_datetime(df_censo["DT_INTER"])
df_censo["saida"] = pd.to_datetime(df_censo["DT_SAIDA"], format="%Y%m%d", errors="coerce")
df_censo["grupo_doenca"] = df_censo["DIAG_PRINC"].astype("string").str.upper().map(classificar_grupo)

dias_2025 = pd.date_range("2025-01-01", "2025-12-31", freq="D")


def censo_diario(base):
    entradas = np.sort(base["entrada"].dropna().values)
    saidas = np.sort(base["saida"].dropna().values)
    internados = [
        np.searchsorted(entradas, dia, side="right") - np.searchsorted(saidas, dia, side="right")
        for dia in dias_2025.values
    ]
    return pd.Series(internados, index=dias_2025)


censo = censo_diario(df_censo)

print("6. Pacientes internados num dia comum (média):", round(censo.mean()))
print("   Maior dia:", censo.idxmax().date(), "com", censo.max(), "internados")

6. Pacientes internados num dia comum (média): 5052
   Maior dia: 2025-05-20 com 5722 internados


In [11]:
censo.groupby(censo.index.month).mean().round(0).rename("internados_por_dia_media").to_frame()

,internados_por_dia_media
1,"4,882.00"
2,"5,107.00"
3,"5,156.00"
4,"5,194.00"
5,"5,250.00"
6,"5,152.00"
7,"5,008.00"
8,"4,991.00"
9,"5,130.00"
10,"5,005.00"


## 6. Nível 2 · O que ocupa o leito (página 2 do painel)

**Pergunta principal:** quais doenças ocupam mais leito, e isso bate com o que custam?

A tabela mostra as três lentes lado a lado: % das internações, % dos leitos-dia e % do valor pago. Quando o % de leitos-dia é maior que o % do valor, o grupo ocupa mais leito do que recebe.

In [12]:
por_grupo = resumo("grupo_doenca")
por_grupo[["internacoes", "leitos_dia", "pct_internacoes", "pct_leitos_dia", "pct_valor",
           "permanencia_media", "rs_por_leito_dia"]].round(1)

,internacoes,leitos_dia,pct_internacoes,pct_leitos_dia,pct_valor,permanencia_media,rs_por_leito_dia
grupo_doenca,,,,,,,
Lesões e traumas,72290,290559,15.80,15.80,13.40,4.00,335.90
Respiratório,45626,230047,10.00,12.50,8.90,5.00,281.00
Circulatório,43907,218573,9.60,11.90,21.00,5.00,698.90
Infecciosas,25529,156559,5.60,8.50,5.90,6.10,275.40
Digestivo,46615,151799,10.20,8.30,7.30,3.30,349.60
Gravidez e parto,59435,145372,13.00,7.90,4.80,2.40,241.00
Geniturinário,35783,134051,7.80,7.30,6.40,3.70,348.00
Câncer,26973,93926,5.90,5.10,10.80,3.50,831.20
Saúde mental,9462,92046,2.10,5.00,0.90,9.70,69.30


In [13]:
por_grupo["descompasso_leito_menos_valor"] = por_grupo["pct_leitos_dia"] - por_grupo["pct_valor"]

print("7. Grupo que mais ocupa leito:", por_grupo.index[0])
print("8. Ocupam mais leito do que recebem:")
print(por_grupo["descompasso_leito_menos_valor"].sort_values(ascending=False).head(3).round(1))
print("9. Recebem mais do que ocupam:")
print(por_grupo["descompasso_leito_menos_valor"].sort_values().head(3).round(1))

7. Grupo que mais ocupa leito: Lesões e traumas
8. Ocupam mais leito do que recebem:
grupo_doenca
Saúde mental       4.10
Respiratório       3.60
Gravidez e parto   3.10
Name: descompasso_leito_menos_valor, dtype: Float64
9. Recebem mais do que ocupam:
grupo_doenca
Circulatório    -9.10
Câncer          -5.60
Osteomuscular   -3.40
Name: descompasso_leito_menos_valor, dtype: Float64


In [14]:
# 10. Procedimentos que mais ocupam leito (nome oficial: consultar a tabela SIGTAP)
por_procedimento = resumo("PROC_REA").head(10)
por_procedimento["diagnostico_mais_comum"] = [
    df.loc[df["PROC_REA"] == proc, "DIAG_PRINC"].value_counts().index[0]
    for proc in por_procedimento.index
]
por_procedimento[["internacoes", "leitos_dia", "pct_leitos_dia", "permanencia_media",
                  "diagnostico_mais_comum"]].round(1)

,internacoes,leitos_dia,pct_leitos_dia,permanencia_media,diagnostico_mais_comum
PROC_REA,,,,,
0303140151,25262,143480,7.80,5.70,J189
0303010037,10436,84876,4.60,8.10,A499
0415010012,21860,83371,4.50,3.80,N201
0303170190,4473,56512,3.10,12.60,F200
0411010034,20023,49070,2.70,2.50,O820
0303060212,5712,39109,2.10,6.80,I500
0303040149,5499,36326,2.00,6.60,I64
0303150050,7832,35430,1.90,4.50,N390
0310010039,15214,34295,1.90,2.30,O800


## 7. Nível 2 · Hospitais (página 3 do painel)

**Pergunta principal:** quais hospitais consomem mais leito, e quanto recebem por dia?

O SIH traz o hospital só pelo código CNES, sem o nome. O painel identifica o hospital pelo código e pelo município.

In [15]:
por_hospital = resumo(["CNES", "nome_municipio_atendimento"])
por_hospital["pct_acumulado"] = por_hospital["pct_leitos_dia"].cumsum()

print("11. Hospitais que mais consomem leito (top 10):")
por_hospital[["internacoes", "leitos_dia", "valor_pago", "rs_por_leito_dia",
              "permanencia_media", "pct_acima_tipico", "pct_acumulado"]].head(10).round(1)

11. Hospitais que mais consomem leito (top 10):


,,internacoes,leitos_dia,valor_pago,rs_por_leito_dia,permanencia_media,pct_acima_tipico,pct_acumulado
CNES,nome_municipio_atendimento,,,,,,,
7743068,Goiânia,28427,148490,"75,583,356.30",509.00,5.20,51.70,8.10
2338262,Goiânia,19352,120561,"35,877,368.50",297.60,6.20,55.90,14.70
0547484,Uruaçu,18639,90241,"23,472,360.40",260.10,4.80,53.50,19.60
2338424,Goiânia,10335,64871,"25,227,175.70",388.90,6.30,56.20,23.10
9680977,Aparecida de Goiânia,10895,59158,"28,399,650.10",480.10,5.40,52.30,26.30
2339196,Goiânia,9871,47942,"13,793,751.80",287.70,4.90,50.50,28.90
2338734,Goiânia,7192,47606,"22,814,965.50",479.20,6.60,55.30,31.50
2506815,Goiânia,12702,45022,"46,749,842.90","1,038.40",3.50,35.90,34.00
0965324,Goiânia,9716,44528,"13,646,151.10",306.50,4.60,56.50,36.40


In [16]:
print("12. Os 10 maiores concentram", round(por_hospital["pct_leitos_dia"].head(10).sum(), 1),
      "% dos leitos-dia, entre", df["CNES"].nunique(), "hospitais")
print("13. R$ por leito-dia entre os 10 maiores: de",
      round(por_hospital["rs_por_leito_dia"].head(10).min()), "a",
      round(por_hospital["rs_por_leito_dia"].head(10).max()))

12. Os 10 maiores concentram 38.7 % dos leitos-dia, entre 260 hospitais
13. R$ por leito-dia entre os 10 maiores: de 260 a 1038


## 8. Nível 2 · Municípios (página 4 do painel)

**Pergunta principal:** quem usa os leitos, onde eles ficam e quem depende de outra cidade?

In [17]:
local_atendimento = df.groupby("nome_municipio_atendimento")["leito_dias"].sum().sort_values(ascending=False)
print("14. Onde ficam os leitos-dia usados (%):")
(local_atendimento / TOTAL_LEITOS_DIA * 100).head(6).round(1)

14. Onde ficam os leitos-dia usados (%):


nome_municipio_atendimento
Goiânia                44.70
Aparecida de Goiânia    7.20
Anápolis                6.80
Uruaçu                  4.90
Rio Verde               4.40
Itumbiara               2.90
Name: leito_dias, dtype: Float64

In [18]:
por_residencia = df.groupby("nome_municipio_residencia").agg(
    internacoes=("N_AIH", "size"),
    leitos_dia=("leito_dias", "sum"),
    valor_pago=("valor_pago", "sum"),
    pct_fora_municipio=("fl_fora_municipio", "mean"),
)
por_residencia["pct_fora_municipio"] *= 100

print("15. Moradores que mais consomem leito:")
por_residencia.sort_values("leitos_dia", ascending=False).head(5).round(1)

15. Moradores que mais consomem leito:


,internacoes,leitos_dia,valor_pago,pct_fora_municipio
nome_municipio_residencia,,,,
Goiânia,78565,389462,"165,924,947.40",7.40
Aparecida de Goiânia,35302,165600,"74,563,111.80",45.40
Anápolis,24797,101311,"50,551,851.70",12.70
Rio Verde,19692,85790,"30,115,123.50",14.20
Senador Canedo,9679,43169,"16,340,731.20",61.20


In [19]:
goiania = df[df["MUNIC_MOV"] == MUNICIPIO_GOIANIA]
pct_fora_goiania = goiania.loc[goiania["fl_fora_municipio"], "leito_dias"].sum() / goiania["leito_dias"].sum() * 100
print("16. % dos leitos-dia de Goiânia usados por moradores de outras cidades:", round(pct_fora_goiania, 1))

print("17. Municípios que mais dependem de outra cidade (3.000+ internações):")
por_residencia[por_residencia["internacoes"] >= 3000].sort_values(
    "pct_fora_municipio", ascending=False
).head(8)[["internacoes", "pct_fora_municipio"]].round(1)

16. % dos leitos-dia de Goiânia usados por moradores de outras cidades: 56.6
17. Municípios que mais dependem de outra cidade (3.000+ internações):


,internacoes,pct_fora_municipio
nome_municipio_residencia,,
Inhumas,4047,67.20
Goianira,6660,66.10
Trindade,9896,63.80
Senador Canedo,9679,61.20
Itaberaí,4077,54.30
Caldas Novas,5171,54.00
Águas Lindas de Goiás,4573,46.30
Aparecida de Goiânia,35302,45.40


In [20]:
print("18. Valor pago pelos moradores de Aparecida de Goiânia (R$):",
      round(por_residencia.loc["Aparecida de Goiânia", "valor_pago"]))

18. Valor pago pelos moradores de Aparecida de Goiânia (R$): 74563112


## 9. Nível 2 · Custo para o SUS (página 5 do painel)

**Pergunta principal:** quanto o SUS paga por cada dia de leito, em cada doença?

R$ por leito-dia é a razão das somas (valor pago ÷ leitos-dia), e não a média das razões de cada internação.

In [21]:
custo = por_grupo[["valor_pago", "rs_por_leito_dia", "rs_por_internacao"]].copy()
print("19. Onde o SUS gasta mais, em reais:")
custo.sort_values("valor_pago", ascending=False).head(5).round(0)

19. Onde o SUS gasta mais, em reais:


,valor_pago,rs_por_leito_dia,rs_por_internacao
grupo_doenca,,,
Circulatório,"152,760,945.00",699.00,"3,479.00"
Lesões e traumas,"97,605,205.00",336.00,"1,350.00"
Câncer,"78,073,854.00",831.00,"2,895.00"
Respiratório,"64,636,592.00",281.00,"1,417.00"
Digestivo,"53,074,438.00",350.00,"1,139.00"


In [22]:
print("20. Internação mais cara (R$ por internação):")
print(custo["rs_por_internacao"].sort_values(ascending=False).head(3).round(0))
print("21-22. R$ por leito-dia, do maior para o menor:")
custo["rs_por_leito_dia"].sort_values(ascending=False).round(0)

20. Internação mais cara (R$ por internação):
grupo_doenca
Malformações    4,416.00
Osteomuscular   3,567.00
Circulatório    3,479.00
Name: rs_por_internacao, dtype: Float64
21-22. R$ por leito-dia, do maior para o menor:


grupo_doenca
Olho e ouvido                   4,776.00
Osteomuscular                   1,040.00
Malformações                      878.00
Câncer                            831.00
Circulatório                      699.00
Outros                            521.00
Contatos com serviço de saúde     430.00
Perinatal                         393.00
Digestivo                         350.00
Geniturinário                     348.00
Lesões e traumas                  336.00
Sintomas mal definidos            322.00
Sistema nervoso                   298.00
Endócrinas                        289.00
Respiratório                      281.00
Infecciosas                       275.00
Gravidez e parto                  241.00
Sangue                            153.00
Pele                              132.00
Saúde mental                       69.00
Name: rs_por_leito_dia, dtype: Float64

In [23]:
print("23. Valor pago por mês de competência (R$ milhões):")
(df.groupby("mes_competencia")["valor_pago"].sum() / 1e6).round(1)

23. Valor pago por mês de competência (R$ milhões):


mes_competencia
1    56.50
2    55.90
3    60.00
4    59.40
5    60.10
6    59.60
7    64.60
8    62.70
9    61.10
10   64.20
11   62.40
12   59.60
Name: valor_pago, dtype: Float64

## 10. Nível 3 · Permanência além do típico (página 6 do painel)

**Pergunta principal:** onde os pacientes ficam mais tempo do que o típico, e quanto isso pesa?

O tempo típico é a mediana do próprio SIH para o mesmo procedimento. A referência oficial seria a permanência média de cada procedimento na tabela SIGTAP (próxima evolução).

In [24]:
total_acima = df["dias_acima_tipico"].sum()
print("24. Leitos-dia acima do típico:", int(total_acima),
      f"({total_acima / TOTAL_LEITOS_DIA * 100:.1f}% do total)")
print("25. Em quais grupos estão esses dias:")
por_grupo[["dias_acima_tipico", "pct_acima_tipico"]].sort_values("dias_acima_tipico", ascending=False).head(5).round(1)

24. Leitos-dia acima do típico: 806293 (43.9% do total)
25. Em quais grupos estão esses dias:


,dias_acima_tipico,pct_acima_tipico
grupo_doenca,,
Lesões e traumas,"141,670.00",48.80
Respiratório,"105,411.50",45.80
Circulatório,"101,367.00",46.40
Digestivo,"66,322.50",43.70
Infecciosas,"65,367.50",41.80


In [25]:
duracao = df.groupby("faixa_duracao", observed=True).agg(
    internacoes=("N_AIH", "size"), leitos_dia=("leito_dias", "sum")
)
duracao["pct_internacoes"] = duracao["internacoes"] / TOTAL_INTERNACOES * 100
duracao["pct_leitos_dia"] = duracao["leitos_dia"] / TOTAL_LEITOS_DIA * 100

longas = df[df["leito_dias"] > LIMITE_INTERNACAO_LONGA]
print(f"26. Internações de mais de {LIMITE_INTERNACAO_LONGA} dias:",
      f"{len(longas) / TOTAL_INTERNACOES * 100:.1f}% das internações e",
      f"{longas['leito_dias'].sum() / TOTAL_LEITOS_DIA * 100:.1f}% dos leitos-dia")
duracao.round(1)

26. Internações de mais de 15 dias: 4.4% das internações e 27.3% dos leitos-dia


,internacoes,leitos_dia,pct_internacoes,pct_leitos_dia
faixa_duracao,,,,
0 dia,48324,0,10.60,0.00
1–3,266401,486034,58.20,26.50
4–7,81653,418460,17.90,22.80
8–15,40761,429536,8.90,23.40
16–30,16891,366462,3.70,20.00
> 30,3373,134735,0.70,7.30


In [26]:
print("27. Hospitais (top 10 em leitos-dia) com mais dias acima do típico:")
print(por_hospital.head(10)["pct_acima_tipico"].sort_values(ascending=False).head(3).round(1))
print("28. Hospital do top 10 que gira o leito mais rápido:")
por_hospital.head(10).sort_values("permanencia_media")[["permanencia_media", "pct_acima_tipico"]].head(1).round(1)

27. Hospitais (top 10 em leitos-dia) com mais dias acima do típico:
CNES     nome_municipio_atendimento
0965324  Goiânia                      56.50
2338424  Goiânia                      56.20
2338262  Goiânia                      55.90
Name: pct_acima_tipico, dtype: Float64
28. Hospital do top 10 que gira o leito mais rápido:


,,permanencia_media,pct_acima_tipico
CNES,nome_municipio_atendimento,,
2506815,Goiânia,3.50,35.90


## 11. Nível 3 · Calendário de ocupação (página 7 do painel)

**Pergunta principal:** quando a rede enche, e em que dia da semana?

In [27]:
por_mes = censo.groupby(censo.index.month).mean()
print("29. Mês com mais internados por dia:", por_mes.idxmax(), "com média de", round(por_mes.max()))

entradas_2025 = df_censo[df_censo["entrada"].dt.year == 2025].groupby(df_censo["entrada"].dt.normalize()).size()
saidas_2025 = df_censo[df_censo["saida"].dt.year == 2025].groupby(df_censo["saida"].dt.normalize()).size()
DIAS_SEMANA = ["seg", "ter", "qua", "qui", "sex", "sáb", "dom"]
semana = pd.DataFrame({
    "entradas_por_dia": entradas_2025.groupby(entradas_2025.index.dayofweek).mean(),
    "altas_por_dia": saidas_2025.groupby(saidas_2025.index.dayofweek).mean(),
}).round(0)
semana.index = DIAS_SEMANA
print("30-31. Entradas e altas por dia da semana:")
semana

29. Mês com mais internados por dia: 5 com média de 5250
30-31. Entradas e altas por dia da semana:


,entradas_por_dia,altas_por_dia
seg,"1,421.00","1,076.00"
ter,"1,435.00","1,339.00"
qua,"1,399.00","1,410.00"
qui,"1,394.00","1,422.00"
sex,"1,279.00","1,467.00"
sáb,936.00,"1,159.00"
dom,943.00,934.00


In [28]:
sazonalidade = {}
for grupo in ["Lesões e traumas", "Respiratório", "Circulatório"]:
    serie = censo_diario(df_censo[df_censo["grupo_doenca"] == grupo])
    mensal = serie.groupby(serie.index.month).mean()
    sazonalidade[grupo] = {
        "media_ano": round(serie.mean()),
        "mes_pico": mensal.idxmax(),
        "pico": round(mensal.max()),
        "mes_vale": mensal.idxmin(),
        "vale": round(mensal.min()),
        "variacao_pico_vs_media_%": round((mensal.max() / serie.mean() - 1) * 100),
    }
print("32. Qual grupo é mais sazonal:")
pd.DataFrame(sazonalidade).T

32. Qual grupo é mais sazonal:


,media_ano,mes_pico,pico,mes_vale,vale,variacao_pico_vs_media_%
Lesões e traumas,792,8,817,1,755,3
Respiratório,624,5,818,12,455,31
Circulatório,609,8,661,1,573,9


## 12. Nível 3 · Simulação de ganho

Quanto se ganharia se os hospitais reduzissem parte dos dias acima do típico? É uma estimativa de capacidade liberada, não uma promessa.

In [29]:
simulacao = pd.DataFrame({"reducao_%": [5, 10, 20, 30]})
simulacao["leitos_dia_liberados"] = total_acima * simulacao["reducao_%"] / 100
simulacao["internacoes_a_mais"] = simulacao["leitos_dia_liberados"] / PERMANENCIA_MEDIA
print("33. Ganho estimado:")
simulacao.round(0)

33. Ganho estimado:


,reducao_%,leitos_dia_liberados,internacoes_a_mais
0,5,"40,315.00","10,048.00"
1,10,"80,629.00","20,096.00"
2,20,"161,259.00","40,191.00"
3,30,"241,888.00","60,287.00"


**34. Onde agir primeiro:** nos procedimentos e hospitais com mais dias acima do típico (seções 7 e 10), nas altas de fim de semana (seção 11) e no preparo para o pico respiratório de maio.

## 13. Curiosidades: como o SUS paga uma internação

Essas contas explicam a dor do produto para o cliente: o SUS paga, na maior parte dos procedimentos, **um pacote fixo** da tabela nacional SIGTAP. Quando o paciente fica mais dias, o valor quase não muda e cada dia a mais pesa no caixa do hospital.

In [30]:
PNEUMONIA = "0303140151"  # tratamento de pneumonias ou influenza (gripe)
pneumonia_sem_uti = df[(df["PROC_REA"] == PNEUMONIA) & (~df["fl_uti"])].copy()
pneumonia_sem_uti["faixa"] = pd.cut(pneumonia_sem_uti["leito_dias"], [-1, 2, 4, 7, 14, np.inf],
                                    labels=["0–2", "3–4", "5–7", "8–14", "15+"])
pacote = pneumonia_sem_uti.groupby("faixa", observed=True).agg(
    internacoes=("N_AIH", "size"),
    valor_medio=("valor_pago", "mean"),
    dias_medios=("leito_dias", "mean"),
)
pacote["rs_por_dia"] = pacote["valor_medio"] / pacote["dias_medios"]
print("Pneumonia sem UTI: o valor sobe pouco e o R$ por dia despenca")
pacote.round(0)

Pneumonia sem UTI: o valor sobe pouco e o R$ por dia despenca


,internacoes,valor_medio,dias_medios,rs_por_dia
faixa,,,,
0–2,6929,606.00,2.00,377.00
3–4,6998,626.00,3.00,183.00
5–7,4565,683.00,6.00,117.00
8–14,2463,839.00,10.00,84.00
15+,758,"1,451.00",23.00,64.00


In [31]:
PROCEDIMENTOS_EXEMPLO = {
    "0310010039": "Parto normal",
    "0411010034": "Parto cesariano",
    "0303140151": "Pneumonia ou gripe",
    "0303170190": "Psiquiatria de curta permanência (paga por dia)",
}
exemplos = df[df["PROC_REA"].isin(PROCEDIMENTOS_EXEMPLO)].groupby("PROC_REA").agg(
    internacoes=("N_AIH", "size"),
    valor_medio=("valor_pago", "mean"),
    dias_medios=("leito_dias", "mean"),
)
exemplos["rs_por_dia"] = exemplos["valor_medio"] / exemplos["dias_medios"]
exemplos.index = exemplos.index.map(PROCEDIMENTOS_EXEMPLO)
print("Quanto custa uma internação:")
print("Metade das internações custou até R$", round(df["valor_pago"].median()))
exemplos.round(0)

Quanto custa uma internação:
Metade das internações custou até R$ 616


,internacoes,valor_medio,dias_medios,rs_por_dia
PROC_REA,,,,
Pneumonia ou gripe,25262,"1,446.00",6.00,255.00
Psiquiatria de curta permanência (paga por dia),4473,"1,032.00",13.00,82.00
Parto normal,15214,579.00,2.00,257.00
Parto cesariano,20023,721.00,2.00,294.00


### 13.1. Valor da UTI

O valor pago pela UTI (`VAL_UTI`) ainda não é extraído para a base silver. Por isso, **só nesta célula**, o notebook lê os arquivos brutos de 2025 direto de `data/raw/sih/2025`. Se os arquivos não estiverem na máquina, a célula apenas avisa.

In [32]:
arquivos_2025 = sorted((PASTA_RAW_SIH / SAFRA).glob("RDGO*.parquet"))

if arquivos_2025:
    uti = pd.concat(
        [pd.read_parquet(a, columns=["IDENT", "VAL_UTI", "VAL_TOT", "UTI_MES_TO"]) for a in arquivos_2025],
        ignore_index=True,
    )
    uti = uti[uti["IDENT"].astype(str) == "1"]
    for coluna in ["VAL_UTI", "VAL_TOT", "UTI_MES_TO"]:
        uti[coluna] = pd.to_numeric(uti[coluna], errors="coerce").fillna(0)
    print("Valor pago pela UTI (R$ milhões):", round(uti["VAL_UTI"].sum() / 1e6, 1))
    print("% do valor total pago:", round(uti["VAL_UTI"].sum() / uti["VAL_TOT"].sum() * 100, 1))
    print("R$ por dia de UTI:", round(uti["VAL_UTI"].sum() / uti["UTI_MES_TO"].sum()))
else:
    print("Arquivos brutos de 2025 não encontrados em", PASTA_RAW_SIH / SAFRA)

Valor pago pela UTI (R$ milhões): 176.1
% do valor total pago: 24.3
R$ por dia de UTI: 631


## 14. Resumo para a venda

Os números-chave do material comercial, calculados neste notebook.

In [33]:
pct = lambda grupo, coluna: round(por_grupo.loc[grupo, coluna], 1)
resumo_venda = pd.DataFrame([
    ("Leitos-dia pagos pelo SUS", f"{TOTAL_LEITOS_DIA:,.0f}"),
    ("Internações", f"{TOTAL_INTERNACOES:,.0f}"),
    ("Valor pago (R$ milhões)", f"{TOTAL_VALOR / 1e6:,.1f}"),
    ("Permanência média (dias)", f"{PERMANENCIA_MEDIA:.1f}"),
    ("R$ por leito-dia", f"{TOTAL_VALOR / TOTAL_LEITOS_DIA:,.0f}"),
    ("Respiratório: % leitos-dia × % valor", f"{pct('Respiratório', 'pct_leitos_dia')} × {pct('Respiratório', 'pct_valor')}"),
    ("Circulatório: % leitos-dia × % valor", f"{pct('Circulatório', 'pct_leitos_dia')} × {pct('Circulatório', 'pct_valor')}"),
    (f"Internações > {LIMITE_INTERNACAO_LONGA} dias: % internações × % leitos-dia",
     f"{len(longas) / TOTAL_INTERNACOES * 100:.1f} × {longas['leito_dias'].sum() / TOTAL_LEITOS_DIA * 100:.1f}"),
    ("% dos leitos-dia acima do típico", f"{total_acima / TOTAL_LEITOS_DIA * 100:.1f}"),
    ("% dos leitos-dia do estado em Goiânia", f"{local_atendimento.get('Goiânia', 0) / TOTAL_LEITOS_DIA * 100:.1f}"),
    ("% dos leitos de Goiânia usados por quem mora fora", f"{pct_fora_goiania:.1f}"),
    ("Internações a mais com 10% menos dias acima do típico",
     f"{simulacao.loc[simulacao['reducao_%'] == 10, 'internacoes_a_mais'].iloc[0]:,.0f}"),
], columns=["indicador", "valor"])
resumo_venda

,indicador,valor
0,Leitos-dia pagos pelo SUS,"1,835,227"
1,Internações,"457,403"
2,Valor pago (R$ milhões),726.1
3,Permanência média (dias),4.0
4,R$ por leito-dia,396
5,Respiratório: % leitos-dia × % valor,12.5 × 8.9
6,Circulatório: % leitos-dia × % valor,11.9 × 21.0
7,Internações > 15 dias: % internações × % leito...,4.4 × 27.3
8,% dos leitos-dia acima do típico,43.9
9,% dos leitos-dia do estado em Goiânia,44.7


## 15. Conclusão e limites

Na safra 2025, o SUS pagou cerca de 1,84 milhão de leitos-dia em Goiás. O painel mostra três pontos que a contagem de internações esconde:

- **Descompasso:** o respiratório ocupa mais leito do que recebe; o circulatório, o contrário.
- **Concentração:** internações de mais de 15 dias são 4,4% dos pacientes e 27% dos leitos-dia.
- **Dependência:** Goiânia concentra 45% dos leitos-dia do estado, e 57% deles são de moradores de outras cidades.

**Limites declarados:**

- O dado chega com 1 a 2 meses de atraso: o produto é de planejamento, não de vaga em tempo real.
- O tempo típico é a mediana do próprio SIH; a referência oficial é a tabela SIGTAP.
- Excluir as AIHs de continuação subestima permanências muito longas, sobretudo em saúde mental.
- O valor pago não é o custo real do hospital.
- Sem o número de leitos existentes (CNES), não há taxa de ocupação.
- Hospitais aparecem pelo código CNES e pelo município: o SIH não tem o nome do hospital, só a coluna `CNES` (código do Cadastro Nacional de Estabelecimentos de Saúde). Os arquivos de leitos do CNES (`LTGO*`) também não têm o nome.